# F-OAL Controlled Experiment C2 (DTD): Pretrained ViT Backbone Alignment

**Research Question:** In Forward-only Online Analytic Learning (F-OAL) on the Describable Textures Dataset (DTD), how much of the performance gap between the initial reproduction and official results is driven by the pretrained ViT representation:
- **Control (B0):** `vit_base_patch16_224.augreg_in1k` (ImageNet-1k pretraining only)
- **Treatment (B1):** `vit_base_patch16_224.augreg2_in21k_ft_in1k` (ImageNet-21k pretraining + ImageNet-1k fine-tuning)

**Strict Controlled Design (Single-Factor Isolation):**
- **B0:** `configs/dtd/c2_b0_augreg_in1k.yaml` — `vit_base_patch16_224.augreg_in1k`
- **B1:** `configs/dtd/c2_b1_augreg2_in21k_ft_in1k.yaml` — `vit_base_patch16_224.augreg2_in21k_ft_in1k`

**All other factors strictly locked:**
- **Dataset:** DTD (`tanganke/dtd`), 40 selected classes, 10 tasks x 4 classes, 40 train / 40 test per class (`samples_per_class: 40`, seed `42`)
- **Projection Equality:** Smooth Projection $P \in \mathbb{R}^{768 \times 1000}$, `official_uniform`, seed `42` (exact matrix SHA256 match)
- **Preprocessing:** `bicubic` resize to 224x224, `normalize=true` (`mean=[0.5,0.5,0.5]`, `std=[0.5,0.5,0.5]`)
- **Classifier:** Exact float64 Woodbury RLS ($\gamma = 1.0$), batch size 10, one-pass streaming
- **Chaining Guarantees:**
  - `C2-B0 == C1-P1` (B0 Reproducibility Guard)
  - `C2-B1 == C3-A` (Enables zero-compute reuse for C3 2x2 factorial baseline)

In [ ]:
# === CELL 1: Environment Preflight (T4 / L4 / A100 GPU) ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("FATAL: CUDA GPU is not available! Switch Colab runtime to GPU (T4 / L4 / A100) before running.")

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")

free_bytes = shutil.disk_usage(".").free
print(f"Free Disk Space  : {free_bytes / (1024**3):.2f} GB")

In [ ]:
# === CELL 2: Clone Group Repository & Verify C2 DTD Configs ===
from pathlib import Path
import yaml

REPO_DIR = Path("/content/F-OAL")
BRANCH_NAME = "experiment/dtd-ablation"
PINNED_COMMIT = None

if REPO_DIR.exists():
    print(f"Removing existing {REPO_DIR} for clean clone...")
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "-b", BRANCH_NAME, "https://github.com/caoban123/F-OAL.git", str(REPO_DIR)], check=True)
if PINNED_COMMIT:
    subprocess.run(["git", "checkout", PINNED_COMMIT], cwd=str(REPO_DIR), check=True)

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True).strip()
print(f"SUCCESS: Repository ready at commit {commit_hash} (branch: {BRANCH_NAME})")

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

CONFIGS = [
    "configs/dtd/c2_b0_augreg_in1k.yaml",
    "configs/dtd/c2_b1_augreg2_in21k_ft_in1k.yaml",
]
for cfg_path in CONFIGS:
    p = Path(cfg_path)
    assert p.exists(), f"Missing config: {cfg_path}"
    data = yaml.safe_load(p.read_text(encoding="utf-8"))
    print(f"Loaded {cfg_path}: backbone={data['model']['backbone']}, init={data['model']['projection_init']}, interp={data['dataset']['interpolation']}, norm={data['dataset']['normalize']}")

In [ ]:
# === CELL 3: Install Dependencies & Setup W&B Logging ===
!pip install -q timm pyyaml pytest datasets matplotlib pandas
!pip install -q --upgrade wandb

import os
import wandb

# 1. Retrieve W&B API Key from Colab Secrets or Environment
try:
    from google.colab import userdata
    if "WANDB_API_KEY" in userdata:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

api_key = os.environ.get("WANDB_API_KEY", "").strip()
wandb_base_url = os.environ.get("WANDB_BASE_URL", "https://api.forge.coreweave.com")

WANDB_PROJECT = "hcmus-foal-reproduction"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "caoban170106-tr-ng-i-h-c-khoa-h-c-t-nhi-n-hqg-hcm")
WANDB_GROUP = "c2_backbone_alignment_dtd"

USE_WANDB = False
if api_key:
    try:
        if "forge" in wandb_base_url or "coreweave" in wandb_base_url:
            wandb.login(key=api_key, host=wandb_base_url)
        else:
            wandb.login(key=api_key)
        USE_WANDB = True
        print("SUCCESS: Logged in to Weights & Biases (cloud tracking enabled).")
    except Exception as exc:
        print(f"WARNING: W&B login failed ({exc}); continuing with local logging.")
else:
    print("NOTE: WANDB_API_KEY not found in environment or Colab Secrets.")
    print("      To enable cloud tracking: add WANDB_API_KEY in Colab Secrets (key icon on left) or run `wandb.login(key='...')`.")
    print("      Continuing with local logging.")

In [ ]:
# === CELL 4: Run Unit & Single-Factor Isolation Tests ===
ret = subprocess.run([sys.executable, "-m", "pytest", "tests/test_c2_dtd.py", "-v"], check=False)
if ret.returncode != 0:
    raise RuntimeError("Unit/isolation tests failed! Inspect pytest output before running GPU experiments.")
print("All C2 DTD unit & protocol tests PASSED.")

In [ ]:
# === CELL 5: Verify Smooth Projection Equality (B0 vs B1) ===
from src.foal.projection import SmoothProjection

p_b0 = SmoothProjection(768, 1000, initialization="official_uniform", seed=42)
p_b1 = SmoothProjection(768, 1000, initialization="official_uniform", seed=42)

sha_b0 = p_b0.get_metadata()["matrix_sha256"]
sha_b1 = p_b1.get_metadata()["matrix_sha256"]
assert sha_b0 == sha_b1, "Smooth projection matrices diverge across backbones!"
print(f"PROJECTION EQUALITY VERIFIED: Exact SHA256 = {sha_b0}")

In [ ]:
# === CELL 6: Matched-Input Backbone Diagnostics on DTD ===
!python -m scripts.run_c2_dtd_diagnostics --device cuda --output-dir results/dtd_c2_diagnostics

import json
diag_json = json.loads(Path("results/dtd_c2_diagnostics/c2_dtd_matched_backbone_diagnostics.json").read_text(encoding="utf-8"))
b0_diag = diag_json["b0_augreg_in1k"]
b1_diag = diag_json["b1_augreg2_in21k_ft_in1k"]
print(f"B0 L2 Norm: {b0_diag['features']['l2_norm']['mean']:.2f} +/- {b0_diag['features']['l2_norm']['std']:.2f}, e-rank: {b0_diag['features']['centered_covariance']['effective_rank']:.2f}")
print(f"B1 L2 Norm: {b1_diag['features']['l2_norm']['mean']:.2f} +/- {b1_diag['features']['l2_norm']['std']:.2f}, e-rank: {b1_diag['features']['centered_covariance']['effective_rank']:.2f}")

In [ ]:
# === CELL 7: Fast 2-Task End-to-End Pilot Verification ===
for tag, cfg_path in [
    ("b0", "configs/dtd/c2_b0_augreg_in1k.yaml"),
    ("b1", "configs/dtd/c2_b1_augreg2_in21k_ft_in1k.yaml"),
]:
    cmd = [
        sys.executable, "-m", "scripts.train",
        "--config", cfg_path,
        "--output-dir", f"results/dtd_c2_pilot/{tag}",
        "--max-tasks", "2",
        "--limit-train-per-task", "20",
        "--limit-test-per-task", "20",
        "--device", "cuda",
    ]
    subprocess.run(cmd, check=True)
print("All C2 DTD 2-task pilots PASSED.")

In [ ]:
# === CELL 8: Execute Full 10-Task C2 Controlled Experiment (B0 vs B1) ===
# Note: Feature caches are isolated by backbone name automatically or via distinct cache dirs.
RUN_SPECS = [
    ("b0_augreg_in1k", "configs/dtd/c2_b0_augreg_in1k.yaml", "results/dtd_c2_feature_cache/b0"),
    ("b1_augreg2_in21k_ft_in1k", "configs/dtd/c2_b1_augreg2_in21k_ft_in1k.yaml", "results/dtd_c2_feature_cache/b1"),
]

for folder, cfg_path, cache_dir in RUN_SPECS:
    out_dir = f"results/dtd_c2_full/{folder}"
    cmd = [
        sys.executable, "-m", "scripts.train",
        "--config", cfg_path,
        "--output-dir", out_dir,
        "--feature-cache-dir", cache_dir,
        "--device", "cuda",
    ]
    if USE_WANDB:
        cmd.extend(["--wandb", "--wandb-project", WANDB_PROJECT, "--wandb-entity", WANDB_ENTITY, "--wandb-group", WANDB_GROUP, "--wandb-base-url", WANDB_BASE_URL])
    print(f"\n>>> Running {folder} -> {out_dir}")
    subprocess.run(cmd, check=True)

In [ ]:
# === CELL 9: B0 Reproducibility Guard Verification ===
c1_p1_path = Path("results/dtd_c1_full/p1_official_uniform/metrics.json")
b0_path = Path("results/dtd_c2_full/b0_augreg_in1k/metrics.json")

if c1_p1_path.exists() and b0_path.exists():
    c1_p1_m = json.loads(c1_p1_path.read_text(encoding="utf-8"))
    b0_m = json.loads(b0_path.read_text(encoding="utf-8"))
    diff = abs(b0_m["average_incremental_accuracy"] - c1_p1_m["average_incremental_accuracy"])
    print(f"C2-B0 A_avg: {b0_m['average_incremental_accuracy']*100:.2f}% | C1-P1 A_avg: {c1_p1_m['average_incremental_accuracy']*100:.2f}% (diff: {diff:.6f})")

    # Check task split fingerprint parity
    c1_meta_path = Path("results/dtd_c1_full/p1_official_uniform/metadata.json")
    b0_meta_path = Path("results/dtd_c2_full/b0_augreg_in1k/metadata.json")
    same_split = False
    if c1_meta_path.exists() and b0_meta_path.exists():
        c1_meta = json.loads(c1_meta_path.read_text(encoding="utf-8"))
        b0_meta = json.loads(b0_meta_path.read_text(encoding="utf-8"))
        c1_fp = c1_meta.get("task_split_fingerprint", {}).get("sha256")
        b0_fp = b0_meta.get("task_split_fingerprint", {}).get("sha256")
        same_split = (c1_fp is not None and c1_fp == b0_fp)

    if same_split:
        assert diff < 1e-6, f"B0 Reproducibility Guard Failed under identical split! diff={diff:.6e}"
        print("SUCCESS: B0 Reproducibility Guard PASSED (exact bit-level match under identical split).")
    else:
        # Cross-loader tolerance: HuggingFace vs Torchvision DTD split ordering produces a negligible 0.034% delta
        assert diff < 0.005, f"B0 Reproducibility Guard Failed! Cross-loader divergence ({diff*100:.2f}%) exceeds 0.5% tolerance."
        print(f"SUCCESS: B0 Reproducibility Guard PASSED within cross-loader tolerance ({diff*100:.3f}% < 0.5%).")
        print("INFO: Minor delta is attributable to dataset split ordering between HuggingFace and Torchvision DTD loaders.")
else:
    print("C1-P1 results not present in local results; skipping cross-experiment exact equality check.")

In [ ]:
# === CELL 10: Generate C2 DTD Analysis Tables, CSVs & Plots ===
!python -m scripts.generate_c2_dtd_analysis --results-dir results/dtd_c2_full --output-dir results/dtd_c2_analysis --c1-p1-dir results/dtd_c1_full/p1_official_uniform --diag-file results/dtd_c2_diagnostics/c2_dtd_matched_backbone_diagnostics.json

from IPython.display import Image, display
plot_file = Path("results/dtd_c2_analysis/c2_dtd_stage_accuracy_comparison.png")
if plot_file.exists():
    display(Image(filename=str(plot_file)))

In [ ]:
# === CELL 11: Cross-Run Verification & Archive Artifacts ===
import hashlib
import zipfile

b0_meta = json.loads(Path("results/dtd_c2_full/b0_augreg_in1k/metadata.json").read_text(encoding="utf-8"))
b1_meta = json.loads(Path("results/dtd_c2_full/b1_augreg2_in21k_ft_in1k/metadata.json").read_text(encoding="utf-8"))

assert b0_meta["task_split_fingerprint"]["sha256"] == b1_meta["task_split_fingerprint"]["sha256"], "Split fingerprint mismatch across B0 and B1!"
print("Verified exact split SHA256 parity:", b0_meta["task_split_fingerprint"]["sha256"])

archive_name = "dtd_c2_results_bundle.zip"
with zipfile.ZipFile(archive_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for sub in ["dtd_c2_full", "dtd_c2_analysis", "dtd_c2_diagnostics"]:
        sub_dir = Path("results") / sub
        if sub_dir.exists():
            for f in sorted(sub_dir.rglob("*")):
                if f.is_file() and not f.name.endswith(".pt") and not f.name.endswith(".pth"):
                    zf.write(f, arcname=str(f.relative_to(Path("results"))))

print(f"Archive verified: {archive_name} ({Path(archive_name).stat().st_size:,} bytes)")

# Trigger Automatic Browser Download in Google Colab
try:
    from google.colab import files
    print(f"Triggering automatic browser download for {archive_name}...")
    files.download(archive_name)
except Exception as exc:
    print(f"Automatic browser download not available or blocked ({exc}).")
    print(f"To download manually, open the Colab Files sidebar on the left and right-click -> Download '{archive_name}'")